In [1]:
!pip install -q --upgrade unsloth trl wandb huggingface_hub

import os
import torch
import wandb
from huggingface_hub import login
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 25.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 58.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 107.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 105.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.1 MB/s eta 0:00:00:0

In [ ]:
# Paste your OWN fresh tokens here — do not reuse any tokens that were ever shared/pasted elsewhere
os.environ["HF_TOKEN"] = "Your_api_key_here"
os.environ["WANDB_API_KEY"] = "your_api_key_here"

login(token=os.environ["HF_TOKEN"])
wandb.login(key=os.environ["WANDB_API_KEY"])

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tesla369 (tesla369-pdepth) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU: Tesla T4
Memory: 15.6 GB


In [3]:
# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------
dataset = load_dataset("TeslaInch/SCD-Instruction-Tuning")

print(dataset)
print(f"\nTrain: {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"\nSample:")
print(dataset['train'][0])

def format_example(example):
    text = f"<|user|>\n{example['instruction']}\n{example['input']}<|end|>\n<|assistant|>\n{example['output']}<|end|>"
    return {"text": text}

train_formatted = dataset['train'].map(format_example)
val_formatted = dataset['validation'].map(format_example)

print(f"Train formatted: {len(train_formatted)}")
print(f"Val formatted: {len(val_formatted)}")
print(f"\nSample formatted:")
print(train_formatted[0]['text'][:500])

README.md:   0%|          | 0.00/465 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.80MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  433kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating validation split:   0%|          | 0/387 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 3487
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 387
    })
})

Train: 3487
Validation: 387

Sample:
{'instruction': 'Answer the following question about sickle cell disease.', 'input': 'What is the prevalence of cardiac enlargement in individuals with sickle cell disease?', 'output': 'Cardiac enlargement, also known as cardiomegaly, is a common complication in individuals with sickle cell disease (SCD). The prevalence of cardiac enlargement in SCD patients varies widely, ranging from 22% to 76% depending on the population studied, the diagnostic criteria used, and the stage of the disease.\n\nThe pathophysiology underlying cardiac enlargement in SCD is multifactorial. Chronic anemia, which is a hallmark of SCD, leads to increased cardiac output to compensate for the reduced oxygen delivery to tissues. This increased wor

Map:   0%|          | 0/3487 [00:00<?, ? examples/s]

Map:   0%|          | 0/387 [00:00<?, ? examples/s]

Train formatted: 3487
Val formatted: 387

Sample formatted:
<|user|>
Answer the following question about sickle cell disease.
What is the prevalence of cardiac enlargement in individuals with sickle cell disease?<|end|>
<|assistant|>
Cardiac enlargement, also known as cardiomegaly, is a common complication in individuals with sickle cell disease (SCD). The prevalence of cardiac enlargement in SCD patients varies widely, ranging from 22% to 76% depending on the population studied, the diagnostic criteria used, and the stage of the disease.

The pathophysi


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="microsoft/Phi-3.5-mini-instruct",
    allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"],
)

In [4]:
# ---------------------------------------------------------------------------
# Model — Unsloth's fast loader (4-bit, patched attention/MLP kernels)
# Your answers are "quite long", so max_seq_length is bumped from 1024 → 2048
# to avoid truncating longer completions. Drop it back to 1024 if you'd
# rather match the original exactly.
# ---------------------------------------------------------------------------
MODEL_ID = "microsoft/Phi-3.5-mini-instruct"  # Unsloth's pre-quantized mirror, loads faster
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="microsoft/Phi-3.5-mini-instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=False,
    use_exact_model_name=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loaded — GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


==((====))==  Unsloth 2026.7.3: Fast Phi3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Unsloth: microsoft/Phi-3.5-mini-instruct had a bad pad_token (<|endoftext|>). Using pad_token = <|placeholder6|>.
Loaded — GPU: 7.65 GB


In [5]:
# ---------------------------------------------------------------------------
# LoRA — same rank/alpha/targets/dropout as your original config
# ---------------------------------------------------------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing (less VRAM, faster)
    random_state=369,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


In [6]:
# ---------------------------------------------------------------------------
# W&B
# ---------------------------------------------------------------------------
wandb.finish()
wandb.init(
    project="scd-medical-llm",
    name="phi35-mini-scd-v8-unsloth",
    config={
        "model": "Phi-3.5-mini-instruct (unsloth)",
        "dataset": "TeslaInch/SCD-Instruction-Tuning",
        "version": "v8",
        "train_examples": len(train_formatted),
        "val_examples": len(val_formatted),
        "lora_rank": 16,
        "lora_alpha": 32,
        "epochs": 3,
        "max_seq_length": MAX_SEQ_LENGTH,
    }
)

wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


In [ ]:
# ---------------------------------------------------------------------------
# Training config — same epochs/batch size/grad accum/LR/schedule/optim
# as your original, just renamed max_length -> max_seq_length for SFTConfig
# ---------------------------------------------------------------------------
sft_config = SFTConfig(
    output_dir="./scd-phi35-adapter-v8",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    learning_rate=1e-4,
    weight_decay=0.05,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
    fp16=True,
    bf16=False,
    optim="adamw_8bit",       # unsloth recommends this over paged_adamw_8bit
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    processing_class=tokenizer,
)

print("Starting training run 8 (Unsloth)...")
print(f"Train examples: {len(train_formatted)}")
print(f"Val examples: {len(val_formatted)}")
print(f"Epochs: 3")
print(f"Steps per epoch: {len(train_formatted) // sft_config.per_device_train_batch_size}")

trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/3487 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/387 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32000}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training run 8 (Unsloth)...
Train examples: 3487
Val examples: 387
Epochs: 3
Steps per epoch: 3487


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,487 | Num Epochs = 3 | Total steps = 1,308
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 3,145,728 of 3,824,225,280 (0.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.561217,0.581237


Unsloth: Restored added_tokens_decoder metadata in ./scd-phi35-adapter-v8/checkpoint-50/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./scd-phi35-adapter-v8/checkpoint-50.


In [ ]:
trainer.model.push_to_hub(
    "TeslaInch/scd-phi35-adapter-v8",
    token=os.environ["your_api_hey_here"],
)
tokenizer.push_to_hub(
    "TeslaInch/scd-phi35-adapter-v8",
    token=os.environ["Your_api_key_here"],
)
wandb.finish()
print("V9 adapter pushed")